In [1]:
# Seuil minimum de confiance acceptable
SEUIL_CONFIANCE = 0.30  # Sous 30%, c'est trop incertain
SEUIL_LONGUEUR_MIN = 5   # Moins de 5 mots = trop court
SEUIL_LONGUEUR_MAX = 500 # Plus de 500 mots = trop long

def detecter_cas_complexe(texte, probas_dict):
    """
    Détecte si une réclamation nécessite un agent humain.
    
    Args:
        texte (str): Le texte de la réclamation
        probas_dict (dict): Dictionnaire {categorie: probabilité}
    
    Returns:
        dict avec:
            - cas_complexe (bool): True si nécessite un humain
            - raisons (list): Liste des raisons
            - confiance_max (float): Probabilité de la meilleure catégorie
    """
    raisons = []
    cas_complexe = False
    
    # Trier les probabilités
    probas_triees = sorted(probas_dict.values(), reverse=True)
    confiance_max = probas_triees[0]
    
    # 1. Confiance trop faible
    if confiance_max < SEUIL_CONFIANCE:
        cas_complexe = True
        raisons.append(f"Confiance trop faible ({confiance_max*100:.1f}%)")
    
    # 2. Texte trop court
    nb_mots = len(str(texte).split())
    if nb_mots < SEUIL_LONGUEUR_MIN:
        cas_complexe = True
        raisons.append(f"Texte trop court ({nb_mots} mots, minimum {SEUIL_LONGUEUR_MIN})")
    
    # 3. Texte très long (suspect)
    if nb_mots > SEUIL_LONGUEUR_MAX:
        cas_complexe = True
        raisons.append(f"Texte très long ({nb_mots} mots)")
    
    # 4. Ambiguïté : 2 catégories très proches
    if len(probas_triees) >= 2:
        diff = probas_triees[0] - probas_triees[1]
        if diff < 0.05:  # moins de 5% d'écart
            cas_complexe = True
            raisons.append(f"Ambiguïté entre catégories (écart de {diff*100:.1f}%)")
    
    return {
        'cas_complexe': cas_complexe,
        'raisons': raisons,
        'confiance_max': confiance_max,
        'necessite_agent': cas_complexe
    }


# Test
test_cases = [
    {
        'texte': "nul",
        'probas': {'mauvaise_qualite': 0.2, 'produit_casse': 0.18, 'erreur_picking': 0.15}
    },
    {
        'texte': "Mon colis n'est jamais arrivé après plusieurs semaines d'attente",
        'probas': {'retard_livraison': 0.85, 'probleme_transport': 0.08, 'erreur_picking': 0.03}
    },
    {
        'texte': "Le produit est cassé et pas conforme",  # ambigu
        'probas': {'produit_casse': 0.32, 'mauvaise_qualite': 0.30, 'erreur_picking': 0.20}
    },
]

for i, cas in enumerate(test_cases, 1):
    result = detecter_cas_complexe(cas['texte'], cas['probas'])
    print(f"\n--- Cas {i} ---")
    print(f"Texte: {cas['texte']}")
    print(f"Cas complexe: {result['cas_complexe']}")
    print(f"Confiance max: {result['confiance_max']*100:.1f}%")
    if result['raisons']:
        print(f"Raisons:")
        for r in result['raisons']:
            print(f"  - {r}")


--- Cas 1 ---
Texte: nul
Cas complexe: True
Confiance max: 20.0%
Raisons:
  - Confiance trop faible (20.0%)
  - Texte trop court (1 mots, minimum 5)
  - Ambiguïté entre catégories (écart de 2.0%)

--- Cas 2 ---
Texte: Mon colis n'est jamais arrivé après plusieurs semaines d'attente
Cas complexe: False
Confiance max: 85.0%

--- Cas 3 ---
Texte: Le produit est cassé et pas conforme
Cas complexe: True
Confiance max: 32.0%
Raisons:
  - Ambiguïté entre catégories (écart de 2.0%)


In [2]:
code_complexity = '''
"""
Module de détection des cas complexes.
"""

SEUIL_CONFIANCE = 0.30
SEUIL_LONGUEUR_MIN = 5
SEUIL_LONGUEUR_MAX = 500


def detecter_cas_complexe(texte, probas_dict):
    """Détecte si une réclamation nécessite un agent humain."""
    raisons = []
    cas_complexe = False
    
    probas_triees = sorted(probas_dict.values(), reverse=True)
    confiance_max = probas_triees[0] if probas_triees else 0
    
    if confiance_max < SEUIL_CONFIANCE:
        cas_complexe = True
        raisons.append(f"Confiance trop faible ({confiance_max*100:.1f}%)")
    
    nb_mots = len(str(texte).split())
    if nb_mots < SEUIL_LONGUEUR_MIN:
        cas_complexe = True
        raisons.append(f"Texte trop court ({nb_mots} mots)")
    
    if nb_mots > SEUIL_LONGUEUR_MAX:
        cas_complexe = True
        raisons.append(f"Texte tres long ({nb_mots} mots)")
    
    if len(probas_triees) >= 2:
        diff = probas_triees[0] - probas_triees[1]
        if diff < 0.05:
            cas_complexe = True
            raisons.append(f"Ambiguite entre categories")
    
    return {
        'cas_complexe': cas_complexe,
        'raisons': raisons,
        'confiance_max': confiance_max,
        'necessite_agent': cas_complexe
    }
'''

with open('../src/complexity.py', 'w', encoding='utf-8') as f:
    f.write(code_complexity)

print("✅ Module sauvegardé dans : src/complexity.py")

✅ Module sauvegardé dans : src/complexity.py
